In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

### Data load

In [8]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

train_id = train['Id']
test_id  = test['Id']

In [11]:
train.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [9]:
y = np.log1p(train['SalePrice'])
train.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

all_data = pd.concat([train, test], axis=0, ignore_index=True)

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 79 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuil

In [16]:
train.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

### Data Cleaning

In [10]:
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType',
             'GarageFinish','GarageQual','GarageCond','BsmtQual','BsmtCond',
             'BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType','MSSubClass']
for c in none_cols:
    all_data[c] = all_data[c].fillna('None')

In [13]:
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2',
             'BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']
for c in zero_cols:
    all_data[c] = all_data[c].fillna(0)

In [14]:
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

In [17]:
cat_cols = all_data.select_dtypes(include='object').columns
for c in cat_cols:
    all_data[c] = all_data[c].fillna(all_data[c].mode()[0])

In [18]:
num_cols = all_data.select_dtypes(exclude='object').columns
for c in num_cols:
    all_data[c] = all_data[c].fillna(all_data[c].median())

### Feature Engineering

In [22]:
# House age feature
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['GarageAge'] = all_data['YrSold'] - all_data['GarageYrBlt']
all_data['IsRemodeled'] = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)
all_data['IsNewHouse'] = (all_data['YearBuilt'] == all_data['YrSold']).astype(int)

# Total area features
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] + all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])
all_data['TotalPorch'] = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] + all_data['3SsnPorch'] + all_data['ScreenPorch'])
all_data['TotalOutdoor'] = all_data['WoodDeckSF'] + all_data['TotalPorch']

In [23]:
qual_map = {'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1,'None':0}
for col in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
            'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']:
    all_data[col+'_num'] = all_data[col].map(qual_map).fillna(0)

all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['QualArea'] = all_data['OverallQual'] * all_data['GrLivArea']
all_data['ExterScore'] = all_data['ExterQual_num'] + all_data['ExterCond_num']
all_data['BsmtScore'] = all_data['BsmtQual_num'] + all_data['BsmtCond_num']
all_data['GarageScore'] = all_data['GarageQual_num'] + all_data['GarageCond_num']
all_data['KitchenScore'] = all_data['KitchenQual_num'] * all_data['OverallQual']

In [25]:
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['Has2ndFloor'] = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)

In [26]:
nbr_stats = all_data.groupby('Neighborhood')['GrLivArea'].agg(['mean','median','std']).reset_index()
nbr_stats.columns = ['Neighborhood','Nbr_AreaMean','Nbr_AreaMedian','Nbr_AreaStd']
all_data = all_data.merge(nbr_stats, on='Neighborhood', how='left')

skewed = ['LotArea','LotFrontage','GrLivArea','TotalSF','TotalBsmtSF',
          '1stFlrSF','MasVnrArea','WoodDeckSF','OpenPorchSF']
for c in skewed:
    all_data[c] = np.log1p(all_data[c])

### Encoding categoricals

In [27]:
cat_cols = all_data.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for c in cat_cols:
    all_data[c] = le.fit_transform(all_data[c].astype(str))

print(f"Total features after engineering: {all_data.shape[1]}")

Total features after engineering: 112


In [28]:
X_train = all_data.iloc[:len(train)].values
X_test  = all_data.iloc[len(train):].values

### Models

In [29]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model, X, y, kf):
    scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=kf)
    return np.sqrt(-scores)

In [30]:
# Gradient Boosting
gbm = GradientBoostingRegressor(
    n_estimators=3000, learning_rate=0.02, max_depth=4,
    max_features='sqrt', min_samples_leaf=15, min_samples_split=10,
    loss='huber', random_state=42
)

In [31]:
# Extra Trees
et = ExtraTreesRegressor(
    n_estimators=1000, max_features=0.5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)

In [32]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=1000, max_features=0.4, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)

In [33]:
# Ridge 
ridge = make_pipeline(RobustScaler(), Ridge(alpha=10.0))

In [34]:
# Lasso
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=10000))

In [ ]:
print("\nCross-validation RMSE scores:")
for name, model in [('GBM', gbm), ('ExtraTrees', et), ('RandomForest', rf), ('Ridge', ridge), ('Lasso', lasso)]:
    scores = rmse_cv(model, X_train, y, kf)
    print(f"  {name:<14}: {scores.mean():.5f} ± {scores.std():.5f}")


Cross-validation RMSE scores:
  GBM           : 0.12234 ± 0.01745
  ExtraTrees    : 0.13363 ± 0.01439
  RandomForest  : 0.13440 ± 0.01711
  Ridge         : 0.14267 ± 0.03019
  Lasso         : 0.14228 ± 0.03043


### Stacking Ensemble

In [36]:
def get_oof_and_test_preds(model, X_train, y, X_test, kf):
    """Out-of-fold predictions for stacking."""
    oof_preds = np.zeros(len(X_train))
    test_preds = np.zeros((len(X_test), kf.n_splits))
    for i, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
        model.fit(X_train[tr_idx], y.iloc[tr_idx])
        oof_preds[val_idx] = model.predict(X_train[val_idx])
        test_preds[:, i] = model.predict(X_test)
    return oof_preds, test_preds.mean(axis=1)

In [ ]:
models = [('GBM', gbm), ('ExtraTrees', et), ('RandomForest', rf), ('Ridge', ridge), ('Lasso', lasso)]

In [38]:
oof_list, test_list = [], []
for name, model in models:
    print(f"  Training {name}...")
    oof, test_p = get_oof_and_test_preds(model, X_train, y, X_test, kf)
    oof_list.append(oof)
    test_list.append(test_p)
    print(f"    OOF RMSE: {np.sqrt(mean_squared_error(y, oof)):.5f}")

  Training GBM...
    OOF RMSE: 0.12358
  Training ExtraTrees...
    OOF RMSE: 0.13440
  Training RandomForest...
    OOF RMSE: 0.13549
  Training Ridge...
    OOF RMSE: 0.14583
  Training Lasso...
    OOF RMSE: 0.14550


In [39]:
oof_stack = np.column_stack(oof_list)
test_stack = np.column_stack(test_list)

In [40]:
meta = Ridge(alpha=0.5)
meta.fit(oof_stack, y)
meta_oof  = meta.predict(oof_stack)
meta_test = meta.predict(test_stack)

In [41]:
print(f"\nEnsemble OOF RMSE: {np.sqrt(mean_squared_error(y, meta_oof)):.5f}")
print(f"Meta-learner weights: {meta.coef_}")


Ensemble OOF RMSE: 0.12329
Meta-learner weights: [ 0.82210381  0.20418325 -0.01932384  0.07208174 -0.06110144]


### Submission

In [43]:
submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': np.expm1(meta_test)
})
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved. Shape: {submission.shape}")
print(submission.describe())


Submission saved. Shape: (1459, 2)
                Id      SalePrice
count  1459.000000    1459.000000
mean   2190.000000  177627.612243
std     421.321334   77234.970935
min    1461.000000   53223.947026
25%    1825.500000  126968.290935
50%    2190.000000  157057.485525
75%    2554.500000  207584.026203
max    2919.000000  558063.351059
